# 02 — CholecSeg8k Class Imbalance Analysis

Quantify the per-class pixel imbalance and validate the imbalance-handling
utilities: inverse-sqrt loss weights and the weighted random sampler.

In [ ]:
# --- Colab setup: install dependencies not preinstalled on Colab ---
# Safe to re-run; skip it if your environment already has these.
%pip install -q datasets "albumentations>=2.0,<3"

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.cholecseg8k import CLASS_NAMES, CholecSeg8kDataset
from src.data.transforms import build_eval_transforms
from src.data.class_balance import (
    compute_pixel_frequencies, class_loss_weights, make_weighted_sampler,
)

## 1. Per-class pixel frequencies

In [ ]:
# Deterministic eval transform -> statistics are not perturbed by augmentation.
train_ds = CholecSeg8kDataset(split="train", transform=build_eval_transforms(512))
counts, freqs = compute_pixel_frequencies(train_ds)

df = pd.DataFrame({"class": CLASS_NAMES, "pixels": counts, "frequency": freqs})
df

## 2. Inverse-sqrt-frequency loss weights (clipped to [0.5, 10.0])

In [ ]:
weights = class_loss_weights(freqs)
for name, w in zip(CLASS_NAMES, weights.tolist()):
    print(f"  {name:14s} weight = {w:6.3f}")

## 3. Weighted random sampler

The sampler oversamples frames containing rare classes. Below, the fraction of
frames containing each class is compared between uniform and weighted sampling.

In [ ]:
sampler = make_weighted_sampler(train_ds)
drawn = list(iter(sampler))[:1000]
baseline = list(range(min(1000, len(train_ds))))


def contains(idx, cls):
    return cls in np.unique(train_ds[idx]["mask"].numpy())


print(f"{'class':14s} {'uniform':>10s} {'weighted':>10s}")
for cls, name in enumerate(CLASS_NAMES):
    base = np.mean([contains(i, cls) for i in baseline])
    samp = np.mean([contains(i, cls) for i in drawn])
    print(f"{name:14s} {base:10.2%} {samp:10.2%}")

## Conclusions

- Rare classes (cystic duct, tool) should appear in a larger fraction of
  weighted-sampled frames than uniformly sampled ones.
- Use `class_loss_weights` for per-class loss weighting and
  `make_weighted_sampler` for the training `DataLoader` (see
  `configs/train/segmentation.yaml`).